# Автоматический подбор музыки для видео

ноутбук запускает весь ML-пайплайн: скачивание подвыборки HarmonySet, подготовка manifest, извлечение признаков, обучение модели, оценка и демо топ-5 рекомендаций

### Установка зависимостей

In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg
!pip -q install opencv-python librosa datasets yt-dlp pandas numpy matplotlib scikit-learn tqdm PyYAML

### Подключение проекта

In [ ]:
import os
import sys
from pathlib import Path

REPO_URL = 'https://github.com/mpaoloo/music-video-retrieval.git'
PROJECT_DIR = Path('/content/music-video-retrieval')

if REPO_URL and not PROJECT_DIR.exists():
    !git clone {REPO_URL} {PROJECT_DIR}

os.chdir(PROJECT_DIR)
sys.path.append(str(PROJECT_DIR))
print('Project directory:', PROJECT_DIR)
print('Files:', os.listdir(PROJECT_DIR)[:10])

## 3. Настройки эксперимента

Для первого запуска лучше поставить `LIMIT = 30` или `LIMIT = 50`, чтобы проверить, что всё работает. Для финального результата можно пробовать `300`, `500` или больше. YouTube-ссылки могут падать, поэтому реально скачанных видео будет меньше, чем `LIMIT`.

In [ ]:
LIMIT = 100

# Если времени мало, оставь 100. Если Colab работает стабильно, попробуй 300-500.
print('Requested subset size:', LIMIT)

## 4. Скачивание данных

HarmonySet хранит YouTube-ссылки и текстовые аннотации. Поэтому сначала скачиваем метаданные и сами видео. Это самая нестабильная часть проекта, потому что часть роликов может быть удалена или недоступна.

In [ ]:
!python -m src.data.download_harmonyset --limit {LIMIT}

## 5. Manifest и простое EDA

Manifest — это таблица только с теми видео, которые реально скачались и открываются через OpenCV. Здесь же делается train/val/test split.

In [ ]:
!python -m src.data.build_manifest

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

manifest = pd.read_csv('data/processed/manifest.csv')
display(manifest.head())
print('Rows:', len(manifest))
print(manifest['split'].value_counts())

plt.figure(figsize=(7, 4))
manifest['duration'].hist(bins=20)
plt.title('Video duration distribution')
plt.xlabel('seconds')
plt.ylabel('count')
plt.tight_layout()
plt.savefig('reports/duration_distribution.png', dpi=160)
plt.show()

## 6. Извлечение признаков

Для видео берём несколько кадров и пропускаем через ResNet18. Для аудио достаём дорожку через ffmpeg и считаем log-mel статистики. Это простые признаки, зато они быстро считаются и легко объясняются.

In [ ]:
!python -m src.features.extract_features

In [ ]:
features = pd.read_csv('data/processed/features/features.csv')
display(features[['pair_id', 'split', 'video_feature_path', 'audio_feature_path']].head())
print('Feature rows:', len(features))
print(features['split'].value_counts())

## 7. Обучение модели

Модель обучается так: в каждом batch есть несколько правильных пар video-audio. Для каждого видео правильная музыка — это аудио с тем же индексом в batch, остальные аудио считаются отрицательными примерами. Это стандартная идея contrastive learning.

In [ ]:
!python -m src.train

In [ ]:
history = pd.read_csv('reports/training_history.csv')
display(history.tail())

plt.figure(figsize=(7, 4))
plt.plot(history['epoch'], history['train_loss'], label='train loss')
plt.plot(history['epoch'], history['val_mrr'], label='val MRR')
plt.legend()
plt.title('Training dynamics')
plt.xlabel('epoch')
plt.tight_layout()
plt.show()

## 8. Оценка качества

Главные метрики: `Hit@1`, `Hit@5`, `MRR`. Также считаем random baseline, чтобы было понятно, лучше ли модель случайного выбора.

In [ ]:
!python -m src.evaluate --split test

In [ ]:
metrics = pd.read_csv('reports/metrics_test.csv')
display(metrics)

metrics.set_index('name')[['hit_at_1', 'hit_at_5', 'mrr']].plot(kind='bar', figsize=(7, 4))
plt.ylim(0, 1)
plt.title('Model vs random baseline')
plt.tight_layout()
plt.show()

## 9. Демо: top-5 рекомендаций

Эту ячейку удобно показывать на защите. Она берёт одно видео из test split и выводит топ-5 аудиодорожек из базы. Если рядом с одной строкой написано `correct pair`, значит модель поставила исходную музыку этого видео в топ.

In [ ]:
!python -m src.infer --split test --row-index 0 --top-k 5

## 10. Что писать в отчёте

Короткая честная интерпретация:

- Проект реализует retrieval baseline для задачи подбора музыки.
- HarmonySet выбран потому, что содержит пары коротких видео и музыки с подробными аннотациями.
- Полный fine-tuning больших MLLM не делался из-за ограничений Colab и срока проекта.
- Модель показывает качество выше/около random baseline; точные числа нужно взять из таблицы выше.
- Главные ограничения: мало данных, простые признаки, слабый учёт временной синхронизации.
- Улучшения: CLIP/CLAP признаки, больше скачанных данных, temporal model, ручная оценка рекомендаций.